# ML Pipeline - Arrest Prediction

**Objective:** Predict arrest likelihood from crime context features using Logistic Regression and Random Forest.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline

spark = SparkSession.builder.appName('CrimeML').enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel('WARN')

In [ ]:
# Load sample data
df = spark.sql('SELECT * FROM team2.crimes_sample')
print('Sample size:', df.count())

# Check class balance
df.groupBy('arrest_flag').count().show()

## 1. Feature Engineering

In [ ]:
# Cyclical encoding for datetime features
df = df.withColumn('hour_sin', sin(col('hour_of_day') * 2 * 3.14159 / 24))
df = df.withColumn('hour_cos', cos(col('hour_of_day') * 2 * 3.14159 / 24))
df = df.withColumn('day_sin', sin(col('day_of_week') * 2 * 3.14159 / 7))
df = df.withColumn('day_cos', cos(col('day_of_week') * 2 * 3.14159 / 7))
df = df.withColumn('month_sin', sin(col('month_of_year') * 2 * 3.14159 / 12))
df = df.withColumn('month_cos', cos(col('month_of_year') * 2 * 3.14159 / 12))

# Index string columns
indexer = StringIndexer(inputCols=['primary_type','iucr','location_description','beat','district','community_area'],
                        outputCols=['primary_type_idx','iucr_idx','loc_desc_idx','beat_idx','district_idx','comm_area_idx'])

model_df = indexer.fit(df).transform(df)
print('Feature engineering complete')

## 2. Assemble Feature Vector

In [ ]:
feature_cols = ['hour_sin','hour_cos','day_sin','day_cos','month_sin','month_cos',
                'primary_type_idx','iucr_idx','loc_desc_idx','beat_idx','district_idx',
                'comm_area_idx','x_coordinate','y_coordinate','latitude','longitude',
                'domestic']

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features', handleInvalid='skip')
assembled = assembler.transform(model_df).select('features', 'arrest_flag')

# Split
train, test = assembled.randomSplit([0.8, 0.2], seed=42)
print(f'Training set: {train.count()}, Test set: {test.count()}')

## 3. Train Models (Logistic Regression + Random Forest + GBT)

In [ ]:
# Logistic Regression
lr = LogisticRegression(featuresCol='features', labelCol='arrest_flag', maxIter=10)
lr_model = lr.fit(train)
lr_pred = lr_model.transform(test)

# Random Forest
rf = RandomForestClassifier(featuresCol='features', labelCol='arrest_flag', numTrees=50)
rf_model = rf.fit(train)
rf_pred = rf_model.transform(test)

# Gradient Boosted Trees
gbt = GBTClassifier(featuresCol='features', labelCol='arrest_flag', maxIter=50)
gbt_model = gbt.fit(train)
gbt_pred = gbt_model.transform(test)

print('All models trained')

## 4. Evaluate Models

In [ ]:
evaluator_auc = BinaryClassificationEvaluator(labelCol='arrest_flag', metricName='areaUnderROC')
evaluator_pr = BinaryClassificationEvaluator(labelCol='arrest_flag', metricName='areaUnderPR')
evaluator_acc = MulticlassClassificationEvaluator(labelCol='arrest_flag', metricName='accuracy')

print('Logistic Regression: AUC-ROC={:.4f}, AUC-PR={:.4f}, Accuracy={:.4f}'.format(
    evaluator_auc.evaluate(lr_pred), evaluator_pr.evaluate(lr_pred), evaluator_acc.evaluate(lr_pred)))

print('Random Forest:      AUC-ROC={:.4f}, AUC-PR={:.4f}, Accuracy={:.4f}'.format(
    evaluator_auc.evaluate(rf_pred), evaluator_pr.evaluate(rf_pred), evaluator_acc.evaluate(rf_pred)))

print('Gradient Boosted:   AUC-ROC={:.4f}, AUC-PR={:.4f}, Accuracy={:.4f}'.format(
    evaluator_auc.evaluate(gbt_pred), evaluator_pr.evaluate(gbt_pred), evaluator_acc.evaluate(gbt_pred)))

## 5. Feature Importance (Random Forest)

In [ ]:
import matplotlib.pyplot as plt

importances = rf_model.featureImportances.toArray()
feat_imp = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)

names = [f[0] for f in feat_imp[:10]]
values = [f[1] for f in feat_imp[:10]]

plt.figure(figsize=(10,5))
plt.barh(names[::-1], values[::-1])
plt.title('Top 10 Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()